In [29]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree
from shapely.geometry import box
import warnings
from pykrige.ok import OrdinaryKriging
warnings.filterwarnings('ignore')

In [19]:
# Adjust these to your actual paths
data_root = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/")

print("=== STATIC LAYER FILES ===\n")

checks = {
    "Vs30 GeoTIFF":     list(data_root.glob("**/*.tif")) + list(data_root.glob("**/*.tiff")),
    "CRUST1.0 files":   list(data_root.glob("**/*.xyz")),
    "DEM GeoTIFFs":     list(data_root.glob("**/dem/*.tif")),
    "Heat flow":        list(data_root.glob("**/*.xlsx")),
    "Stress map":       list(data_root.glob("**/*.csv")),
    "GEM faults":       list(data_root.glob("**/*.shp")),
    "Tectonic plates":  list(data_root.glob("**/*.json")),
}

for layer, files in checks.items():
    print(f"{layer}:")
    if files:
        for f in files:
            print(f"  {f}")
    else:
        print(f"  NOT FOUND")
    print()

=== STATIC LAYER FILES ===

Vs30 GeoTIFF:
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P06_NewZealand_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P01_Kanto_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P12_Ordos_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P07_Sumatra_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P04_Turkey_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P02_Tohoku_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P10_Australia_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/P09_Longmenshan_dem.tif
  /Users/rahulravi23/Desktop/Work/seismic_hazard_mo

In [20]:
# Compare the two
original   = gpd.read_file("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/active_faults_shapefile/gem_active_faults.shp")
harmonized = gpd.read_file("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/active_faults_shapefile/gem_active_faults_harmonized.shp")

print("=== ORIGINAL ===")
print(f"Rows: {len(original)}")
print(f"Columns: {original.columns.tolist()}")
print(f"Slip types ({original['slip_type'].nunique()} unique):")
print(original['slip_type'].value_counts().head(10))

print("\n=== HARMONIZED ===")
print(f"Rows: {len(harmonized)}")
print(f"Columns: {harmonized.columns.tolist()}")
# Check what the slip type column is called
slip_col = [c for c in harmonized.columns if 'slip' in c.lower()]
print(f"Slip-related columns: {slip_col}")
if slip_col:
    print(f"Slip types ({harmonized[slip_col[0]].nunique()} unique):")
    print(harmonized[slip_col[0]].value_counts().head(10))

=== ORIGINAL ===
Rows: 16195
Columns: ['WKT_GEOMET', 'accuracy', 'activity_c', 'average_di', 'average_ra', 'catalog_id', 'catalog_na', 'dip_dir', 'downthrown', 'downthro_1', 'epistemic_', 'exposure_q', 'fs_name', 'is_active', 'last_movem', 'lower_seis', 'name', 'net_slip_r', 'notes', 'ogc_fid', 'reference', 'shortening', 'slip_type', 'strike_sli', 'upper_seis', 'vert_sep_r', 'geometry']
Slip types (23 unique):
slip_type
Reverse                2960
Normal                 2874
Spreading_Ridge        1876
Subduction_Thrust      1499
Dextral                1226
Sinistral              1001
Sinistral_Transform     581
Dextral_Transform       581
Strike-Slip             575
Reverse-Strike-Slip     406
Name: count, dtype: int64

=== HARMONIZED ===
Rows: 13696
Columns: ['average_di', 'average_ra', 'catalog_id', 'catalog_na', 'dip_dir', 'lower_seis', 'name', 'net_slip_r', 'slip_type', 'upper_seis', 'reference', 'epistemic_', 'accuracy', 'activity_c', 'fs_name', 'last_movem', 'downthrown', 'vert_

In [21]:
raster_dir = "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/rasters/"

PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Target grid resolution
RES = 0.1  # degrees

In [22]:
VS30_PATH = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/vs30/vs30_mosaic.tif")

print(f"{'Patch':<25} {'Shape':>12} {'Min':>8} {'Max':>8} {'Mean':>8} {'NaN%':>8}")
print("-" * 70)

with rasterio.open(VS30_PATH) as src:
    for patch_name, b in PATCHES.items():
        # Define target grid
        lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
        lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
        n_lon, n_lat = len(lons), len(lats)

        # Read window from source at native resolution
        window = from_bounds(
            b['minlon'], b['minlat'],
            b['maxlon'], b['maxlat'],
            src.transform
        )
        data_native = src.read(1, window=window).astype(float)

        # Resample to 0.1 degree grid using average
        from rasterio.transform import from_bounds as tfrom_bounds
        target_transform = tfrom_bounds(
            b['minlon'], b['minlat'],
            b['maxlon'], b['maxlat'],
            n_lon, n_lat
        )

        # Use rasterio reproject for clean resampling
        from rasterio.warp import reproject, Resampling as RS
        output = np.zeros((n_lat, n_lon), dtype=np.float32)

        reproject(
            source=data_native,
            destination=output,
            src_transform=src.window_transform(window),
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=src.crs,
            resampling=RS.average
        )

        # Handle nodata
        output = output.astype(float)
        output[output <= 0] = np.nan

        # Save as numpy array
        np.save(raster_dir + f"{patch_name}_vs30.npy", output)
        np.save(raster_dir + f"{patch_name}_lons.npy", lons)
        np.save(raster_dir + f"{patch_name}_lats.npy", lats)

        valid = output[~np.isnan(output)]
        nan_pct = 100 * np.isnan(output).sum() / output.size

        print(f"{patch_name:<25} {str(output.shape):>12} "
              f"{valid.min():>8.1f} {valid.max():>8.1f} "
              f"{valid.mean():>8.1f} {nan_pct:>7.1f}%")

print("\nSaved Vs30 rasters to data/rasters/")

Patch                            Shape      Min      Max     Mean     NaN%
----------------------------------------------------------------------
Kanto_Japan                   (27, 30)    159.3    776.0    532.5     0.0%
Tohoku_Japan                  (30, 30)    198.6    776.0    586.6     0.0%
Central_Chile                 (30, 30)    219.4    878.1    622.3     0.0%
Central_Turkey                (25, 35)    200.9    847.8    521.3     0.0%
Central_Nepal                 (27, 30)    211.4    896.3    690.9     0.0%
North_Island_NZ               (30, 35)    176.2   1002.0    765.3     0.0%
Sumatra                       (35, 40)    190.2    810.8    487.6     0.0%
Kutch_India                   (30, 35)    184.1    799.6    343.4     0.0%
Sichuan_China                 (30, 35)    224.4    894.4    640.4     0.0%
W_Australia                   (30, 35)    206.2    901.8    441.4     0.0%
S_Norway                      (30, 40)    384.7    900.0    826.9     0.0%
Ordos_China                  

In [23]:
dem_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem/")

# Map patch names to DEM filenames
DEM_FILES = {
    "Kanto_Japan":     "P01_Kanto_dem.tif",
    "Tohoku_Japan":    "P02_Tohoku_dem.tif",
    "Central_Chile":   "P03_Chile_dem.tif",
    "Central_Turkey":  "P04_Turkey_dem.tif",
    "Central_Nepal":   "P05_Nepal_dem.tif",
    "North_Island_NZ": "P06_NewZealand_dem.tif",
    "Sumatra":         "P07_Sumatra_dem.tif",
    "Kutch_India":     "P08_Kutch_dem.tif",
    "Sichuan_China":   "P09_Longmenshan_dem.tif",
    "W_Australia":     "P10_Australia_dem.tif",
    "S_Norway":        "P11_Norway_dem.tif",
    "Ordos_China":     "P12_Ordos_dem.tif",
}

PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Patches where negative values are real terrain (keep as-is)
KEEP_NEGATIVES = {"Kutch_India"}

print(f"{'Patch':<25} {'Shape':>12} {'Min m':>8} {'Max m':>8} {'Mean m':>8} {'NaN%':>8}")
print("-" * 75)

for patch_name, b in PATCHES.items():
    dem_path = dem_dir / DEM_FILES[patch_name]

    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
    n_lon, n_lat = len(lons), len(lats)

    with rasterio.open(dem_path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata

        # Handle nodata
        if nodata is not None:
            data[data == nodata] = np.nan

        # Clip negatives to zero except where real terrain
        if patch_name not in KEEP_NEGATIVES:
            data[data < 0] = 0.0

        target_transform = tfrom_bounds(
            b['minlon'], b['minlat'],
            b['maxlon'], b['maxlat'],
            n_lon, n_lat
        )

        output = np.zeros((n_lat, n_lon), dtype=np.float32)
        output[:] = np.nan

        reproject(
            source=data.astype(np.float32),
            destination=output,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=src.crs,
            resampling=Resampling.average
        )

    output = output.astype(float)

    # Compute slope from resampled DEM
    # Simple gradient-based slope in degrees
    dy, dx = np.gradient(output, RES * 111000, RES * 111000)  # convert deg to metres
    slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

    # Roughness: std of elevation in 3x3 neighbourhood
    from scipy.ndimage import generic_filter
    roughness = generic_filter(output, np.nanstd, size=3)

    # Save
    np.save(raster_dir + f"{patch_name}_dem.npy",       output)
    np.save(raster_dir + f"{patch_name}_slope.npy",     slope)
    np.save(raster_dir + f"{patch_name}_roughness.npy", roughness)

    valid = output[~np.isnan(output)]
    nan_pct = 100 * np.isnan(output).sum() / output.size

    print(f"{patch_name:<25} {str(output.shape):>12} "
          f"{np.nanmin(output):>8.1f} {np.nanmax(output):>8.1f} "
          f"{np.nanmean(output):>8.1f} {nan_pct:>7.1f}%")

print("\nSaved DEM, slope, roughness rasters to data/rasters/")

Patch                            Shape    Min m    Max m   Mean m     NaN%
---------------------------------------------------------------------------


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Kanto_Japan                   (27, 30)      0.0   2051.6    314.5    21.6%


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Tohoku_Japan                  (30, 30)      0.0   1196.6    213.8    38.9%


...

Sumatra                       (35, 40)      0.0   2197.6    203.0     8.9%


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Kutch_India                   (30, 35)     -2.2    289.9     44.6     2.4%
Sichuan_China                 (30, 35)    298.1   4544.4   1788.7     0.0%
W_Australia                   (30, 35)    225.9    518.8    383.7     0.0%


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


S_Norway                      (30, 40)      0.0   1610.7    665.5    16.7%
Ordos_China                   (30, 35)    750.4   1737.4   1274.3     0.0%

Saved DEM, slope, roughness rasters to data/rasters/


In [26]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

def load_xyz_as_grid(filepath, value_col='value', scale=1.0):
    """Load XYZ file and reshape into a regular grid for interpolation."""
    df = pd.read_csv(filepath, sep='\s+', header=None,
                     names=['lon', 'lat', value_col])
    df[value_col] = df[value_col] * scale

    # CRUST1.0 is a regular 1-degree grid
    lons_grid = np.sort(df['lon'].unique())
    lats_grid = np.sort(df['lat'].unique())

    # Reshape to 2D grid (lat x lon)
    grid = df.pivot(index='lat', columns='lon', values=value_col).values

    return lats_grid, lons_grid, grid

# Load both CRUST1.0 layers
print("Loading CRUST1.0 layers...")
sed_lats, sed_lons, sed_grid = load_xyz_as_grid(
    "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/sedthk-m.xyz",
    value_col='sediment',
    scale=1/1000  # metres to km
)
crs_lats, crs_lons, crs_grid = load_xyz_as_grid(
    "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/crsthk.xyz",
    value_col='crustal',
    scale=1.0  # already in km
)
print(f"Sediment grid shape: {sed_grid.shape}")
print(f"Crustal grid shape:  {crs_grid.shape}")

# Build interpolators
sed_interp = RegularGridInterpolator(
    (sed_lats, sed_lons), sed_grid,
    method='linear', bounds_error=False, fill_value=np.nan
)
crs_interp = RegularGridInterpolator(
    (crs_lats, crs_lons), crs_grid,
    method='linear', bounds_error=False, fill_value=np.nan
)

print(f"\n{'Patch':<25} {'Shape':>12} "
      f"{'Sed min':>8} {'Sed max':>8} {'Sed mean':>9} "
      f"{'Crs min':>8} {'Crs max':>8} {'Crs mean':>9}")
print("-" * 90)

for patch_name, b in PATCHES.items():
    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)

    # Build query points (lat, lon) pairs
    grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
    points = np.column_stack([grid_lats.ravel(), grid_lons.ravel()])

    sed = sed_interp(points).reshape(len(lats), len(lons))
    crs = crs_interp(points).reshape(len(lats), len(lons))

    # Clip negatives
    sed = np.clip(sed, 0, None)
    crs = np.clip(crs, 0, None)

    np.save(raster_dir + f"{patch_name}_sediment.npy",  sed)
    np.save(raster_dir + f"{patch_name}_crustal.npy",   crs)

    print(f"{patch_name:<25} {str(sed.shape):>12} "
          f"{np.nanmin(sed):>8.2f} {np.nanmax(sed):>8.2f} {np.nanmean(sed):>9.2f} "
          f"{np.nanmin(crs):>8.2f} {np.nanmax(crs):>8.2f} {np.nanmean(crs):>9.2f}")

print("\nSaved sediment and crustal thickness rasters to data/rasters/")

Loading CRUST1.0 layers...
Sediment grid shape: (180, 360)
Crustal grid shape:  (180, 360)

Patch                            Shape  Sed min  Sed max  Sed mean  Crs min  Crs max  Crs mean
------------------------------------------------------------------------------------------
Kanto_Japan                   (27, 30)     0.15     2.38      0.89    11.71    31.30     26.07
Tohoku_Japan                  (30, 30)     0.05     1.86      0.66    11.97    29.81     23.00
Central_Chile                 (30, 30)     0.06     2.74      0.39     9.48    53.98     41.43
Central_Turkey                (25, 35)     0.00     3.97      0.64    33.05    41.68     37.13
Central_Nepal                 (27, 30)     0.00     4.86      0.85    42.50    71.48     54.38
North_Island_NZ               (30, 35)     0.21     3.71      1.44    15.44    40.09     30.49
Sumatra                       (35, 40)     0.36     3.42      1.68     7.83    33.00     27.32
Kutch_India                   (30, 35)     0.01     3.79 

In [28]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Slip type consolidation map
SLIP_MAP = {
    'Dextral': 'Strike_Slip', 'Sinistral': 'Strike_Slip',
    'Dextral_Transform': 'Strike_Slip', 'Sinistral_Transform': 'Strike_Slip',
    'Strike-Slip': 'Strike_Slip',
    'Reverse': 'Reverse', 'Subduction_Thrust': 'Reverse',
    'Reverse-Strike-Slip': 'Reverse', 'Dextral-Reverse': 'Reverse',
    'Sinistral-Reverse': 'Reverse', 'Reverse-Dextral': 'Reverse',
    'Reverse-Sinistral': 'Reverse', 'Anticline': 'Reverse',
    'Blind Thrust': 'Reverse',
    'Normal': 'Normal', 'Normal-Dextral': 'Normal',
    'Dextral-Normal': 'Normal', 'Sinistral-Normal': 'Normal',
    'Normal-Sinistral': 'Normal', 'Normal-Strike-Slip': 'Normal',
    'Spreading_Ridge': 'Spreading',
    'Dextral-Oblique': 'Oblique', 'Syncline': 'Oblique',
}
SLIP_ENCODE = {'Strike_Slip': 0, 'Reverse': 1, 'Normal': 2,
               'Spreading': 3, 'Oblique': 4, 'Unknown': 5}

# Australia weight multiplier for GEM Faulted Earth traces
AUS_WEIGHT = 0.5

print("Loading GEM active faults...")
faults = gpd.read_file(
    "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/active_faults_shapefile/gem_active_faults.shp"
).set_crs("EPSG:4326")
faults['slip_consolidated'] = faults['slip_type'].map(SLIP_MAP).fillna('Unknown')
print(f"Loaded {len(faults)} fault traces")

print(f"\n{'Patch':<25} {'Shape':>12} "
      f"{'dist min':>9} {'dist max':>9} {'dist mean':>10} "
      f"{'density mean':>13} {'dom slip':>12}")
print("-" * 95)

for patch_name, b in PATCHES.items():
    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
    grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
    cell_coords = np.column_stack([grid_lons.ravel(), grid_lats.ravel()])

    # Clip faults to patch + 1 degree buffer
    bbox = box(b['minlon']-1, b['minlat']-1,
               b['maxlon']+1, b['maxlat']+1)
    local = faults[faults.intersects(bbox)].copy()

    # ── Distance to nearest fault ─────────────────────────────────────
    if len(local) > 0:
        # Sample vertices from all fault traces
        fault_pts = []
        for geom in local.geometry:
            if geom is None:
                continue
            if geom.geom_type == 'MultiLineString':
                for line in geom.geoms:
                    fault_pts.extend(list(line.coords))
            elif geom.geom_type == 'LineString':
                fault_pts.extend(list(geom.coords))
        fault_pts = np.array(fault_pts)  # (N, 2) lon/lat

        tree = cKDTree(fault_pts)
        dist_deg, _ = tree.query(cell_coords)
        dist_km = dist_deg * 111.0
    else:
        # No faults — stable craton/rebound setting
        dist_km = np.full(len(cell_coords), 999.0)

    # ── Fault density (km of fault per 100x100 km cell) ───────────────
    density = np.zeros(len(cell_coords))
    if len(local) > 0:
        local_proj = local.to_crs("EPSG:3857")
        # Apply weight for GEM Faulted Earth traces in Australia
        if patch_name == "W_Australia":
            weights = np.where(
                local['catalog_na'] == 'GEM Faulted Earth',
                AUS_WEIGHT, 1.0
            )
        else:
            weights = np.ones(len(local))

        for i, (lon, lat) in enumerate(cell_coords):
            # 0.5 degree radius (~55 km)
            nearby_mask = (
                (local.geometry.bounds['minx'] < lon + 0.5) &
                (local.geometry.bounds['maxx'] > lon - 0.5) &
                (local.geometry.bounds['miny'] < lat + 0.5) &
                (local.geometry.bounds['maxy'] > lat - 0.5)
            )
            nearby_idx = np.where(nearby_mask)[0]
            if len(nearby_idx) > 0:
                lengths_km = local_proj.geometry.iloc[nearby_idx].length.values / 1000
                density[i] = np.sum(lengths_km * weights[nearby_idx]) / (110 * 0.5)**2

    # ── Dominant slip type ────────────────────────────────────────────
    slip_grid = np.full(len(cell_coords), SLIP_ENCODE['Unknown'])
    if len(local) > 0:
        for i, (lon, lat) in enumerate(cell_coords):
            nearby_mask = (
                (local.geometry.bounds['minx'] < lon + 0.5) &
                (local.geometry.bounds['maxx'] > lon - 0.5) &
                (local.geometry.bounds['miny'] < lat + 0.5) &
                (local.geometry.bounds['maxy'] > lat - 0.5)
            )
            nearby = local[nearby_mask]
            if len(nearby) > 0:
                dom = nearby['slip_consolidated'].mode()[0]
                slip_grid[i] = SLIP_ENCODE.get(dom, 5)

    # Reshape
    shape = (len(lats), len(lons))
    dist_raster    = dist_km.reshape(shape)
    density_raster = density.reshape(shape)
    slip_raster    = slip_grid.reshape(shape)

    # Save
    np.save(raster_dir + f"{patch_name}_dist_fault.npy",    dist_raster)
    np.save(raster_dir + f"{patch_name}_fault_density.npy", density_raster)
    np.save(raster_dir + f"{patch_name}_fault_slip.npy",    slip_raster)

    dom_slip = [k for k, v in SLIP_ENCODE.items()
                if v == int(np.median(slip_raster))][0]

    print(f"{patch_name:<25} {str(dist_raster.shape):>12} "
          f"{dist_raster.min():>9.1f} {dist_raster.max():>9.1f} "
          f"{dist_raster.mean():>10.1f} "
          f"{density_raster.mean():>13.4f} "
          f"{dom_slip:>12}")

print("\nSaved fault features to data/rasters/")

Loading GEM active faults...
Loaded 16195 fault traces

Patch                            Shape  dist min  dist max  dist mean  density mean     dom slip
-----------------------------------------------------------------------------------------------
Kanto_Japan                   (27, 30)       0.2     114.1       31.1        0.0756      Reverse
Tohoku_Japan                  (30, 30)       0.2     164.1       74.2        0.0317      Unknown
Central_Chile                 (30, 30)       2.2     219.9       90.4        2.4449      Reverse
Central_Turkey                (25, 35)       0.2     108.1       27.0        0.3544  Strike_Slip
Central_Nepal                 (27, 30)       0.7      85.8       32.9        0.1159       Normal
North_Island_NZ               (30, 35)       0.1      83.9       16.9        0.3122       Normal
Sumatra                       (35, 40)       2.1     264.9       83.5        0.0733  Strike_Slip
Kutch_India                   (30, 35)       2.7     269.9      101.2   

In [32]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Literature values for patches with no data
LITERATURE_HF = {
    "Central_Nepal": 75.0,   # Tanaka et al. (2004) — central Himalayan foreland
}

# Buffer expansion for sparse patches (degrees)
BUFFER = {
    "Kutch_India":  3.0,
    "W_Australia":  2.0,
    "S_Norway":     3.0,
    "Ordos_China":  3.0,
}

# Load and clean heat flow data
print("Loading heat flow data...")
# df = pd.read_excel(
#     "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Heat_Flow_DB/IHFC_2024_GHFDB_v.2026.03.xlsx",
#     sheet_name="GHFDB R20024 v.2026.03", header=None
# )
# df.columns = df.iloc[4]
# df = df.iloc[5:].reset_index(drop=True)

xl = pd.ExcelFile("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Heat_Flow_DB/IHFC_2024_GHFDB_v.2026.03.xlsx")
print(f"Sheets: {xl.sheet_names}")

df = pd.read_excel("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Heat_Flow_DB/IHFC_2024_GHFDB_v.2026.03.xlsx",sheet_name=xl.sheet_names[1],header=5)
print(f"Shape: {df.shape}")

df['q']       = pd.to_numeric(df['q'],       errors='coerce')
df['lat_NS']  = pd.to_numeric(df['lat_NS'],  errors='coerce')
df['long_EW'] = pd.to_numeric(df['long_EW'], errors='coerce')

# Quality filter: 0-500 mW/m², lower bound 20
df_clean = df[
    (df['q'] >= 20) &
    (df['q'] <= 500) &
    df['lat_NS'].notna() &
    df['long_EW'].notna()
].copy()
print(f"Clean heat flow points: {len(df_clean):,}")

print(f"\n{'Patch':<25} {'N pts':>7} {'Method':>12} "
      f"{'Min':>7} {'Max':>7} {'Mean':>7} {'NaN%':>7}")
print("-" * 75)

for patch_name, b in PATCHES.items():
    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
    shape = (len(lats), len(lons))

    # Get buffer for this patch
    buf = BUFFER.get(patch_name, 0.5)

    # Extract points within patch + buffer
    pts = df_clean[
        (df_clean['long_EW'] >= b['minlon'] - buf) &
        (df_clean['long_EW'] <= b['maxlon'] + buf) &
        (df_clean['lat_NS']  >= b['minlat'] - buf) &
        (df_clean['lat_NS']  <= b['maxlat'] + buf)
    ].copy()

    hf_raster     = np.zeros(shape)
    hf_mask       = np.zeros(shape, dtype=int)  # 0=kriged, 1=literature, 2=unavailable
    method_used   = 'kriging'

    # ── Literature imputation ─────────────────────────────────────────
    if patch_name in LITERATURE_HF:
        hf_raster[:] = LITERATURE_HF[patch_name]
        hf_mask[:]   = 1
        method_used  = 'literature'
        n_pts        = 0

    # ── Kriging ───────────────────────────────────────────────────────
    elif len(pts) >= 5:
        try:
            ok = OrdinaryKriging(
                pts['long_EW'].values,
                pts['lat_NS'].values,
                pts['q'].values,
                variogram_model='exponential',
                verbose=False,
                enable_plotting=False,
            )
            z, ss = ok.execute(
                'grid',
                lons,
                lats,
            )
            hf_raster  = np.array(z)
            # Clip to physical range
            hf_raster  = np.clip(hf_raster, 20, 500)
            hf_mask[:] = 0
            method_used = 'kriging'
            n_pts = len(pts)

        except Exception as e:
            print(f"  Kriging failed for {patch_name}: {e}")
            hf_raster[:] = pts['q'].mean()
            hf_mask[:]   = 2
            method_used  = 'mean_fallback'
            n_pts = len(pts)

    # ── Too sparse — use regional mean ───────────────────────────────
    elif len(pts) > 0:
        hf_raster[:] = pts['q'].mean()
        hf_mask[:]   = 2
        method_used  = 'mean_fallback'
        n_pts        = len(pts)

    else:
        hf_raster[:] = np.nan
        hf_mask[:]   = 2
        method_used  = 'unavailable'
        n_pts        = 0

    np.save(raster_dir + f"{patch_name}_heatflow.npy",      hf_raster)
    np.save(raster_dir + f"{patch_name}_heatflow_mask.npy", hf_mask)

    valid    = hf_raster[~np.isnan(hf_raster)]
    nan_pct  = 100 * np.isnan(hf_raster).sum() / hf_raster.size

    print(f"{patch_name:<25} {n_pts:>7} {method_used:>12} "
          f"{np.nanmin(hf_raster):>7.1f} {np.nanmax(hf_raster):>7.1f} "
          f"{np.nanmean(hf_raster):>7.1f} {nan_pct:>6.1f}%")

print("\nSaved heat flow rasters to data/rasters/")

Loading heat flow data...
Sheets: ['Metadata', 'GHFDB R20024 v.2026.03']
Shape: (91182, 67)
Clean heat flow points: 86,200

Patch                       N pts       Method     Min     Max    Mean    NaN%
---------------------------------------------------------------------------
Kanto_Japan                   283      kriging    72.8   112.6    84.2    0.0%
Tohoku_Japan                  329      kriging    38.2   156.9    66.2    0.0%
Central_Chile                  52      kriging    46.7   231.7   136.9    0.0%
Central_Turkey                 50      kriging    37.5   183.2    71.6    0.0%
Central_Nepal                   0   literature    75.0    75.0    75.0    0.0%
North_Island_NZ               469      kriging   110.4   110.9   110.9    0.0%
Sumatra                        66      kriging    30.2   118.4    76.7    0.0%
Kutch_India                   124      kriging    68.0    80.8    76.0    0.0%
Sichuan_China                  47      kriging    46.6    63.1    55.9    0.0%
W_Australi

In [33]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Regime encoding
REGIME_ENCODE = {
    'TF': 0,   # Thrust faulting
    'SS': 1,   # Strike-slip
    'NF': 2,   # Normal faulting
    'NS': 3,   # Normal + strike-slip
    'TS': 4,   # Thrust + strike-slip
    'U':  5,   # Undetermined
}

# Patches needing buffer expansion
BUFFER = {
    "Central_Nepal": 4.0,
    "W_Australia":   3.0,
    "S_Norway":      3.0,
    "Ordos_China":   3.0,
    "Kutch_India":   2.0,
}

# Patches where regime is unreliable — use azimuth only
REGIME_UNRELIABLE = {"Ordos_China", "S_Norway"}

print("Loading World Stress Map...")
df = pd.read_csv(
    "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/tectonic_stress/WSM_Database_2025.csv",
    low_memory=False
)

# Quality filter A-C, valid azimuth
df_clean = df[
    df['QUALITY'].isin(['A', 'B', 'C']) &
    (pd.to_numeric(df['AZI'], errors='coerce') > 0) &
    (pd.to_numeric(df['AZI'], errors='coerce') < 360)
].copy()
df_clean['AZI'] = pd.to_numeric(df_clean['AZI'])
print(f"Clean WSM points: {len(df_clean):,}")

print(f"\n{'Patch':<25} {'N pts':>7} {'Method':>14} "
      f"{'AZI min':>8} {'AZI max':>8} {'AZI mean':>9} "
      f"{'Dom regime':>11}")
print("-" * 85)

for patch_name, b in PATCHES.items():
    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
    shape = (len(lats), len(lons))

    buf = BUFFER.get(patch_name, 0.5)

    pts = df_clean[
        (df_clean['LON'] >= b['minlon'] - buf) &
        (df_clean['LON'] <= b['maxlon'] + buf) &
        (df_clean['LAT'] >= b['minlat'] - buf) &
        (df_clean['LAT'] <= b['maxlat'] + buf)
    ].copy()

    azi_raster    = np.zeros(shape)
    regime_raster = np.full(shape, REGIME_ENCODE['U'])
    azi_mask      = np.zeros(shape, dtype=int)
    method_used   = 'kriging'

    # ── Azimuth kriging ───────────────────────────────────────────────
    # Azimuth is circular (0-360) — use sin/cos decomposition
    if len(pts) >= 5:
        try:
            azi_rad = np.radians(pts['AZI'].values)
            sin_azi = np.sin(azi_rad)
            cos_azi = np.cos(azi_rad)

            # Krige sin and cos components separately
            ok_sin = OrdinaryKriging(
                pts['LON'].values, pts['LAT'].values, sin_azi,
                variogram_model='spherical', verbose=False,
                enable_plotting=False
            )
            ok_cos = OrdinaryKriging(
                pts['LON'].values, pts['LAT'].values, cos_azi,
                variogram_model='spherical', verbose=False,
                enable_plotting=False
            )
            z_sin, _ = ok_sin.execute('grid', lons, lats)
            z_cos, _ = ok_cos.execute('grid', lons, lats)

            # Reconstruct azimuth from sin/cos
            azi_reconstructed = np.degrees(np.arctan2(
                np.array(z_sin),
                np.array(z_cos)
            ))
            # Convert to 0-360 range
            azi_raster = azi_reconstructed % 360
            method_used = 'kriging'

        except Exception as e:
            print(f"  Kriging failed for {patch_name}: {e}")
            azi_raster[:] = pts['AZI'].mean()
            azi_mask[:]   = 1
            method_used   = 'mean_fallback'

    elif len(pts) > 0:
        azi_raster[:] = pts['AZI'].mean()
        azi_mask[:]   = 1
        method_used   = 'mean_fallback'

    else:
        azi_raster[:] = np.nan
        azi_mask[:]   = 2
        method_used   = 'unavailable'

    # ── Dominant regime per cell ──────────────────────────────────────
    if patch_name not in REGIME_UNRELIABLE and len(pts) >= 5:
        grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
        for i in range(shape[0]):
            for j in range(shape[1]):
                cell_lon = grid_lons[i, j]
                cell_lat = grid_lats[i, j]
                nearby = pts[
                    (np.abs(pts['LON'] - cell_lon) < 0.5) &
                    (np.abs(pts['LAT'] - cell_lat) < 0.5)
                ]
                if len(nearby) > 0:
                    dom = nearby['REGIME'].mode()[0]
                    regime_raster[i, j] = REGIME_ENCODE.get(dom, 5)

    # Save
    np.save(raster_dir + f"{patch_name}_stress_azi.npy",    azi_raster)
    np.save(raster_dir + f"{patch_name}_stress_regime.npy", regime_raster)
    np.save(raster_dir + f"{patch_name}_stress_mask.npy",   azi_mask)

    dom_regime_code = int(np.median(regime_raster))
    dom_regime = [k for k, v in REGIME_ENCODE.items()
                  if v == dom_regime_code][0]

    print(f"{patch_name:<25} {len(pts):>7} {method_used:>14} "
          f"{np.nanmin(azi_raster):>8.1f} {np.nanmax(azi_raster):>8.1f} "
          f"{np.nanmean(azi_raster):>9.1f} {dom_regime:>11}")

print("\nSaved stress rasters to data/rasters/")

Loading World Stress Map...
Clean WSM points: 77,127

Patch                       N pts         Method  AZI min  AZI max  AZI mean  Dom regime
-------------------------------------------------------------------------------------
Kanto_Japan                  5929        kriging     17.0    154.2     110.2          SS
Tohoku_Japan                 4343        kriging     26.6    148.0      94.7          TF
Central_Chile                 316        kriging     17.1    154.8      82.7          SS
Central_Turkey                290        kriging      5.0    169.0      61.2          SS
Central_Nepal                 178        kriging     15.5    168.0      63.3          NF
North_Island_NZ               378        kriging     39.2    138.2      65.5          NF
Sumatra                       198        kriging      7.0     84.5      40.4          SS
Kutch_India                    56        kriging     13.0    166.2      42.1           U
Sichuan_China                 126        kriging     66.1  